<a href="https://colab.research.google.com/github/Alexandraqq1/RetuRO-Data-Analysis/blob/main/RetuRO_ETL_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pdfplumber
import pandas as pd

folder = "/content/"
toate_datele = []
setari = {"vertical_strategy": "text", "horizontal_strategy": "text"}

for fisier in os.listdir(folder):
    if fisier.endswith(".pdf"):
        cale = os.path.join(folder, fisier)

        # Extragem anul din numele PDF-ului
        anul = "Necunoscut"
        if "24" in fisier: anul = "2024"
        elif "25" in fisier: anul = "2025"
        elif "26" in fisier: anul = "2026"

        with pdfplumber.open(cale) as pdf:
            for pagina in pdf.pages:
                tabel = pagina.extract_table(setari)
                if not tabel: continue

                df = pd.DataFrame(tabel)
                if len(df.columns) < 8: continue

                df = df.iloc[:, :8]
                df.columns = ["C0", "C1", "PB", "PK", "MB", "MK", "SB", "SK"]

                # Curățăm absolut toate virgulele și spațiile din numere dintr-un foc
                for col in ["PB", "PK", "MB", "MK", "SB", "SK"]:
                    df[col] = df[col].astype(str).str.replace(r'[, ]', '', regex=True)

                # FILTRUL MAGIC: Păstrăm DOAR rândurile care au efectiv un număr la Plastic_Buc
                df = df[df['PB'].str.isnumeric()]

                # Reparăm decalajul dintre 2024 și anii noi
                if anul == "2024":
                    df['Luna'] = df['C0']
                    df['Judet'] = df['C1']
                else:
                    df['Luna'] = fisier.replace(".pdf", "") # Luăm luna din titlul fișierului
                    df['Judet'] = df['C0']

                df['Anul'] = anul

                # Aranjăm coloanele frumos și salvăm
                df_final = df[['Anul', 'Luna', 'Judet', 'PB', 'PK', 'MB', 'MK', 'SB', 'SK']].copy()
                df_final.columns = ["Anul", "Luna", "Judet", "Plastic_Buc", "Plastic_Kg", "Metal_Buc", "Metal_Kg", "Sticla_Buc", "Sticla_Kg"]
                toate_datele.append(df_final)

if toate_datele:
    tabel_suprem = pd.concat(toate_datele, ignore_index=True)
    # SALVĂM TOTUL ÎN EXCEL
    tabel_suprem.to_excel("/content/Date_RetuRO_Curatate.xlsx", index=False)
    print("GATA! Am curățat datele și am creat fișierul Excel.")
    display(tabel_suprem.head(30))
else:
    print("Eroare: Nu am putut extrage datele.")

GATA! Am curățat datele și am creat fișierul Excel.


,Anul,Luna,Judet,Plastic_Buc,Plastic_Kg,Metal_Buc,Metal_Kg,Sticla_Buc,Sticla_Kg
0,2025,Septembrie 2025,Alba,3836289,127120,2018896,28428,1666279,373528
1,2025,Septembrie 2025,Arad,6933511,234114,3097750,42576,1873609,415051
2,2025,Septembrie 2025,Argeș,7741184,262074,3968005,55780,3137816,776172
3,2025,Septembrie 2025,Bacău,5725657,201927,4039972,57252,2197553,623649
4,2025,Septembrie 2025,Bihor,7971540,271267,4395877,60264,2320519,542851
5,2025,Septembrie 2025,Bistrița-Năsăud,2387605,81725,1821672,25685,1653708,354864
6,2025,Septembrie 2025,Botoșani,3477972,123953,2214279,31312,1083644,291664
7,2025,Septembrie 2025,Brăila,3350424,119817,1831560,25322,1186757,313459
8,2025,Septembrie 2025,Brașov,7668300,251936,4316737,61301,3180834,850186
9,2025,Septembrie 2025,București,34913475,1144997,15438030,216519,14289871,4005276
